# Hour 1 · Corpora and data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/1a_corpora_and_data.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F1a_corpora_and_data.ipynb)


**Goal:** see how a clay tablet becomes a *programmatically accessible corpus* — a graph of objects (tablet → column → line → word → sign) with features.

We use the **Copenhagen Ugaritic Corpus (CUC)**, served as JSONL from HuggingFace and loaded with one function. Licence: **CC BY-NC 4.0** (educational use).

// TODO: DO WE WANT TO SWITCH TO TF CUC?

# 2. Excavations, tablets, and corpora as data

*Hour 1 · ~15 min · notebook `1a`*

> **Status:** outline stub.

> **New term?** *Corpus, Text-Fabric, "graph of objects", JSONL* and other
> computational terms are unpacked in plain language in [glossary.md](glossary.md).

## Excavations and archives
- Excavations: **1928-1939**, resumed **1950-2008**. The Ras Shamra mound was
  identified after remains were exposed near Minet el-Beida; French excavations
  under Claude F.A. Schaeffer began in **1929**.
- Ugarit's best-documented phase is the Late Bronze Age, c. **1450-1200 BCE**:
  royal palaces, temples, shrines, an acropolis library, family/private archives,
  and administrative records.
- Royal, temple, palatial, and private **archives and libraries** preserve clay
  tablets in alphabetic Ugaritic cuneiform and other scripts/languages.
- The tablets cover political, social, economic, religious, and cultural life,
  which is why Ugarit works so well as a compact DH corpus.

*Background source: Encyclopaedia Britannica,
["Ugarit"](https://www.britannica.com/place/Ugarit).*

## One tablet, many forms
A single tablet exists at once as a **museum object, photograph, transliteration,
translation, commentary, dictionary references, catalogue entry, corpus record,
and bibliography.**

> **Central idea:** the DH task *begins* with integrating these scattered
> representations into normalized, queryable data.

![KTU 1.1, first tablet of the Baal Myth, Louvre AO 16641.](../images/0000215274_OG.JPG)

*Figure: **KTU 1.1**, the first tablet of the Baal Myth. Source:
[Louvre collections](https://collections.louvre.fr/en/ark:/53355/cl010141446).
Image © 2004 GrandPalaisRmn (musée du Louvre) / Franck Raux.*

This is the same object seen through several lenses: museum object, photograph,
KTU number, literary label, digital corpus page, transliteration, and eventually
tokens in a notebook. A corpus is useful precisely because it keeps those layers
linked.

## From book to corpus
A digital corpus is **not an e-book**. It is a **graph of objects and features**:
text → line → word → morphological unit, each carrying features (form, lemma,
part of speech, …).

## The main resources (detail in `../data/README.md`)
- **CUC — Cuneiform Ugaritic Corpus**: original Text-Fabric dataset, 278 KTU
  tablets (CACCHT), plus the `AlexWalhai/cuc` HuggingFace JSONL mirror used by
  the notebooks. It is a work-in-progress digital corpus from the CACCHT project
  (Christian Canu Højgaard, Martijn Naaijer, Martin Ehrensvärd, Robert Rezetko,
  Oliver Glanz, Willem van Peursen), licensed **CC BY-NC 4.0**.
- **CUC coverage:** 278 KTU tablets, currently in ranges **KTU 1.1-1.180**,
  **KTU 2.1-2.113**, and **KTU 3.1-3.35**; not every number in those ranges is
  present.
- **CUC features:** tablet, column, line, side, consonantal word form
  (`g_cons`), word divider/trailer, language, sign, emendation (`emen`),
  certainty (`cert`), line continuation (`cont`), and alternative reading
  (`alt`).
- **ContextFabric**: graph corpus engine + `cfabric-mcp` server for LLMs/agents.
- **UDB — Ugaritic Data Bank**: Spanish-team corpus of CAT texts, mostly under
  the same numbers; now a licensed Accordance package, with UDB/concordance PDFs
  listed on Juan-Pablo Vita's Academia page.
- **USC Digital Library / InscriptiFact**: high-resolution tablet photographs
  from Bruce Zuckerman and the West Semitic Research Project; formerly
  `inscriptifact.com`, now under USC Digital Library access/reuse terms.
- **KTU**: standard text numbering; **DUL/DULAT**: the standard dictionary.
- For comparison: **BHSA** (Hebrew Bible), **DSS** (Dead Sea Scrolls), and other
  Text-Fabric / ContextFabric corpora.

## TODO
- [ ] Add a diagram: "one tablet → nine representations".
- [ ] Add a screenshot of a Text-Fabric corpus structure.

## 0. Setup


In [10]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. One tablet, two scripts
Every line carries both a Latin transliteration (`lines`) and the actual **cuneiform** (`ugaritic`), plus a reference (`refs`).

In [11]:
t = [x for x in texts if x["ktu"] == "1.3"][0]   # Baal Cycle
for ref, lat, cun in zip(t["refs"][:6], t["lines"][:6], t["ugaritic"][:6]):
    print(f'{ref:14s} {lat:28s} {cun}')

KTU 1.3 I 1    al . tġl[ . ]bd[xxxxx]       𐎀𐎍 𐎟 𐎚𐎙𐎍[ 𐎟 ]𐎁𐎄[xxxxx]
KTU 1.3 I 2    p rdmn . ʿbd . ali[yn]       𐎔 𐎗𐎄𐎎𐎐 𐎟 𐎓𐎁𐎄 𐎟 𐎀𐎍𐎛[𐎊𐎐]
KTU 1.3 I 3    bʿl . sid . zbl . bʿl        𐎁𐎓𐎍 𐎟 𐎒𐎛𐎄 𐎟 𐎇𐎁𐎍 𐎟 𐎁𐎓𐎍
KTU 1.3 I 4    arṣ . qm . yṯʿr              𐎀𐎗𐎕 𐎟 𐎖𐎎 𐎟 𐎊𐎘𐎓𐎗
KTU 1.3 I 5    w . yšlḥmnh                  𐎆 𐎟 𐎊𐎌𐎍𐎈𐎎𐎐𐎅
KTU 1.3 I 6    ybr d . ṯd . l pnwh          𐎊𐎁𐎗 𐎄 𐎟 𐎘𐎄 𐎟 𐎍 𐎔𐎐𐎆𐎅


## 2. The corpus is a graph of object types
In Text-Fabric the CUC objects are **tablet · column · line · word · sign**. Our loader keeps tablets, lines, word tokens, and signs.

In [12]:
print("tablets :", len(texts))
print("lines   :", sum(len(t["lines"]) for t in texts))
print("words   :", sum(len(t["tokens"]) for t in texts))
from data.loader import sign_counts
print("signs   :", sum(sign_counts(texts).values()))

tablets : 278
lines   : 7577
words   : 25477
signs   : 70490


## 3. Browse by genre
Genre labels are heuristic (KTU number + known tablets).

In [13]:
from data.loader import texts_by_genre
for g, items in sorted(texts_by_genre(texts).items(), key=lambda kv: -len(kv[1])):
    print(f'{g:20s} {len(items):3d}  e.g. {", ".join(x["ktu"] for x in items[:5])}')

letter               104  e.g. 2.1, 2.10, 2.100, 2.101, 2.102
literary/religious    90  e.g. 1.101, 1.104, 1.107, 1.108, 1.111
legal/economic        35  e.g. 3.1, 3.10, 3.11, 3.12, 3.13
ritual                15  e.g. 1.105, 1.106, 1.109, 1.112, 1.119
myth                  12  e.g. 1.1, 1.100, 1.114, 1.2, 1.23
divination            10  e.g. 1.103, 1.124, 1.140, 1.141, 1.142
epic                   9  e.g. 1.14, 1.15, 1.16, 1.17, 1.18
god-list               3  e.g. 1.102, 1.118, 1.47


## 4. Zoom out — the whole excavated corpus

The 278 CUC tablets above are the **digitised, transliterated** slice we can compute on directly. But Ugarit's excavations yielded **thousands** of inscribed objects. Two community catalogues let us see the shape of that whole find:

- the **USC / West Semitic Research** *Ugaritic Texts* catalogue (`data/ugaritic_texts_catalog.tsv`), and
- the **Ugarit Texts Database** (F. Válek, *Digital Religion at Ugarit*; `data/UGARIT_TEXTS_DATABASE.csv` + `data/analyse.py`).

Reusing that project's own normalisation, let's chart the corpus by **genre**, **language**, and **discovery archive** — the questions every corpus project asks first.

*Sources: [USC/WSRP Ugaritic Texts](https://dornsife.usc.edu/wsrp-temp/wp-content/uploads/sites/155/2023/05/Ugaritic-Texts.pdf) · [DigitalReligion-Ugarit](https://github.com/valekfrantisek/DigitalReligion-Ugarit).*

In [14]:
import re
import pandas as pd
from data import analyse                 # DigitalReligion-Ugarit normalisation helpers

db = pd.read_csv("../data/UGARIT_TEXTS_DATABASE.csv", encoding="utf-8",
                 delimiter=";", dtype=str, index_col=0)

def ktu_genre(k):                         # KTU number -> genre, by its leading digit
    if not isinstance(k, str): return None
    m = re.match(r"\d+", k.strip().split(".")[0])
    return analyse.ktu_classification.get(int(m.group())) if m else None

genre    = db["KTU3"].map(ktu_genre).dropna().value_counts()
language = db["UTDB Language"].map(lambda x: analyse.normalise_lang(x, use_multilingual=True)).value_counts()
archive  = db["Archive/General area"].fillna(db["SAU Archive/General area"]).fillna("Other/unknown").value_counts()

print(f"{len(db)} excavated text-objects in the database "
      f"(vs the {len(texts)} transliterated tablets in CUC).\n")
print("By genre (KTU class):\n", genre.to_string(), "\n")
print("By language:\n", language.head(8).to_string(), "\n")
print("By discovery archive (top 10):\n", archive.head(10).to_string())

4582 excavated text-objects in the database (vs the 278 transliterated tablets in CUC).

By genre (KTU class):
 KTU3
Economic                  843
Unclassified etc.         694
Literary and Religious    177
Letters                   108
Inscriptions              106
Legal and Juridical        35
Scribal Excercises         34
Ugaritic in syllabic        1 

By language:
 UTDB Language
Ugaritic              1928
Akkadian              1708
Sumerian               285
multilingual           227
unknown/unassigned     191
Egyptian               108
Hurrian                 98
Cypro-Minoan            16 

By discovery archive (top 10):
 Archive/General area
Royal Palace                     1689
House of Urtenu                   683
House of Rapanu                   491
Other/unknown                     354
Ras Ibn-Hani                      193
House of the Hurrian Priest       167
House of the High Priest          151
House of Yabninu                  141
Lamaštu                           104


In [15]:
import plotly.express as px

def bar(series, title, color, n=12):
    d = series.head(n)[::-1]                              # largest on top
    fig = px.bar(x=d.values, y=d.index, orientation="h", title=title,
                 text=d.values, color_discrete_sequence=[color])
    fig.update_traces(textposition="outside")
    fig.update_layout(height=380, width=760, xaxis_title="texts", yaxis_title="",
                      margin=dict(l=210, r=40, t=50, b=30),
                      paper_bgcolor="white", plot_bgcolor="white")
    fig.show()

bar(genre,    "Ugarit corpus by genre (KTU classification)", "#4c78a8")
bar(language, "Ugarit corpus by language", "#e07a5f")
bar(archive,  "Ugarit corpus by discovery archive", "#7fb285")

### Where was each language and genre found?

The marginals above hide the *joint* picture. Cross-tabulating **discovery archive × language** and **discovery archive × genre** reveals the social geography of writing at Ugarit — Akkadian diplomatic correspondence massed in the **House of Urtenu**, literary & religious texts in the **priests' houses**, economic tablets everywhere but concentrated in the **Royal Palace**. (`Latin` / `Phoenician` are scribal-script labels, dropped from the language view following `analyse.py`.)

In [16]:
import plotly.express as px

meta = pd.DataFrame({
    "archive": db["Archive/General area"].fillna(db["SAU Archive/General area"]).fillna("Other/unknown"),
    "language": db["UTDB Language"].map(lambda x: analyse.normalise_lang(x, use_multilingual=True))
                  .replace("", "unknown/unassigned"),
    "genre": db["KTU3"].map(ktu_genre),
})
TOP = meta["archive"].value_counts().head(10).index
meta["arch"] = meta["archive"].where(meta["archive"].isin(TOP), "Other")

def stacked(col, title, drop=(), order=None):
    ct = pd.crosstab(meta["arch"], meta[col]).drop(columns=list(drop), errors="ignore")
    ct = ct.loc[ct.sum(axis=1).sort_values().index]                 # archives by total
    long = ct.reset_index().melt(id_vars="arch", var_name=col, value_name="texts")
    kw = {"category_orders": {col: order}} if order else {}
    fig = px.bar(long, x="texts", y="arch", color=col, orientation="h", title=title, **kw)
    fig.update_layout(barmode="stack", height=470, width=900, yaxis_title="", xaxis_title="texts",
                      legend_title_text=col, margin=dict(l=215, r=30, t=50, b=40),
                      paper_bgcolor="white", plot_bgcolor="white")
    fig.show()

lang_order = [l for l in analyse.order_of_languages if l not in ("Latin", "Phoenician")] + ["multilingual"]
stacked("language", "Discovery archive × language", drop=["Latin", "Phoenician"], order=lang_order)
stacked("genre",    "Discovery archive × genre (KTU classification)")

**What the whole corpus looks like — three facts the 278-tablet sample hides:**

- **It's mostly bureaucracy.** *Economic / administrative* tablets (~840) dwarf the famous *literary & religious* texts (~180): the Baal Cycle and Kirta are a tiny, glamorous minority of what was actually written.
- **It's genuinely bilingual.** **Ugaritic ≈ Akkadian** in sheer volume, with Hurrian, Sumerian, Egyptian, Hittite, and Cypro-Minoan all present — Ugarit's "crossroads" status, quantified (cf. `docs/01`).
- **Writing clusters by household.** The **Royal Palace** holds the largest archive, but named private houses (**Urtenu**, **Rapanu**, the **High** and **Hurrian Priests**) each kept substantial collections — the social geography of literacy.

So the corpus we model in the rest of the workshop is a *curated, literary* window onto a much larger, more workaday, multilingual archive.

## 5. Simple queries
A corpus lets us *ask questions* in code.

In [17]:
# find every tablet that contains a given word form
TARGET = "mlk"   # "king"
hits = [t["ktu"] for t in texts if TARGET in t["tokens"]]
print(f'{TARGET!r} appears in {len(hits)} tablets:', hits[:15], "...")

'mlk' appears in 89 tablets: ['1.1', '1.100', '1.103', '1.105', '1.106', '1.107', '1.108', '1.109', '1.111', '1.112', '1.115', '1.119', '1.126', '1.132', '1.139'] ...


In [18]:
# print every line of one tablet that contains the word
ktu = "1.4"
one = [t for t in texts if t["ktu"] == ktu][0]
for ref, line in zip(one["refs"], one["lines"]):
    if TARGET in line.replace(".", " ").split():
        print(f'{ref}: {line}')

KTU 1.4 I 5: [il . abh . i]l mlk
KTU 1.4 II 44: mlk
KTU 1.4 III 9: [xxx]xy . ilm . d mlk
KTU 1.4 IV 24: qrš . mlk . ab . šnnm
KTU 1.4 IV 38: dm . ʿṣm . hm . yd . il mlk
KTU 1.4 IV 48: [i]l . mlk . d yknnh . yṣḥ
KTU 1.4 VII 43: u mlk . u bl mlk
KTU 1.4 VIII 49: [spr . ilmlk . ṯ]ʿy . nqmd . mlk . ugrt


## 6. Discussion
A corpus is a **graph of objects and features**, queryable in code. CUC shares the Text-Fabric model with **BHSA** (Hebrew Bible) and **DSS** — the same queries port across corpora.

> **With full Text-Fabric** (`pip install text-fabric`; `use('DT-UCPH/cuc')`) you also get sign-level features: emendation, certainty, alternative readings.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [19]:
# Change the word and re-run. Try: "ilm" (gods), "ank" (I), "bʿl" (Baal), "ytn" (he gave)
MY_WORD = "ilm"
tablets = [t["ktu"] for t in texts if MY_WORD in t["tokens"]]
print(f'{MY_WORD!r} occurs in {len(tablets)} tablets:', tablets[:20])
# Hint: words are case-sensitive transliteration — ʿ ʾ š ḥ ṯ ġ are distinct letters.

'ilm' occurs in 69 tablets: ['1.1', '1.103', '1.104', '1.105', '1.106', '1.107', '1.112', '1.114', '1.117', '1.118', '1.124', '1.132', '1.133', '1.134', '1.139', '1.147', '1.15', '1.16', '1.17', '1.176']
